# Лабораторная работа  
## Обучение модели обнаружения объектов с YOLO26 на собственных данных

**Основа:** туториал Roboflow *How to Train a YOLO26 Object Detection Model with Custom Data*  
**Источник:** https://blog.roboflow.com/how-to-train-yolo26-custom-data/

---

### Цель работы
Освоить полный цикл обучения модели обнаружения объектов на собственном датасете:
- самостоятельно собрать и разметить изображения в **Roboflow**;
- экспортировать датасет в формате, совместимом с YOLO;
- обучить модель **YOLO26**;
- сравнить качество и скорость обучения для разных **весов модели**;
- исследовать влияние **гиперпараметров** обучения;
- выполнить инференс и проанализировать ошибки модели.

### Краткая опора на исходный туториал
В статье Roboflow показан базовый пайплайн:
1. включить **GPU** в Colab;
2. установить пакеты `ultralytics`, `supervision`, `roboflow`;
3. скачать датасет через Roboflow API;
4. обучить модель командой вида  
   `yolo task=detect mode=train model=yolo26m.pt data=... epochs=20 imgsz=640 plots=True`;
5. использовать файл `best.pt` для инференса и визуализации результатов.

В данной лабораторной работе этот базовый сценарий расширяется:
- датасет размечается **самостоятельно**;
- проводится сравнение **нескольких предобученных весов**;
- проводится серия экспериментов с **параметрами обучения**.

## 1. Требования к результату

По завершении работы студент должен предоставить:
1. заполненный данный notebook;
2. ссылку на проект в **Roboflow**;
3. обученные веса лучшей модели;
4. краткий отчет с таблицей экспериментов и выводами;
5. примеры удачных и неудачных детекций.

## 2. Исходные условия и ограничения

### Требования к датасету
Выберите **одну прикладную задачу обнаружения объектов**. Примеры:
- кружки / бутылки / телефоны;
- автомобили / велосипеды / самокаты;
- лица / каски / рюкзаки;
- книги / клавиатуры / мыши;
- детали оборудования или любые другие понятные объекты.

### Минимальные требования к данным
- не менее **3 классов** объектов;
- не менее **120 изображений**;
- не менее **40 изображений на класс** суммарно;
- данные должны быть **размечены самостоятельно** в Roboflow;
- желательно обеспечить разнообразие:
  - ракурсов;
  - масштаба;
  - освещения;
  - частичных перекрытий;
  - фона.

### Разделение выборки
Рекомендуемое разбиение:
- train: **70%**
- valid: **20%**
- test: **10%**

## 3. Задание

### Часть A. Подготовка данных
1. Соберите собственный набор изображений для задачи обнаружения объектов.
2. Загрузите изображения в **Roboflow**.
3. Выполните **ручную разметку bounding boxes** для всех объектов.
4. Проверьте качество разметки:
   - нет ли пропущенных объектов;
   - не выходят ли боксы далеко за границы объекта;
   - нет ли перепутанных классов.
5. Сгенерируйте версию датасета и экспортируйте ее в формат, совместимый с **YOLO**.

### Часть B. Базовое обучение
1. Подготовьте окружение.
2. Подключите датасет из Roboflow.
3. Обучите базовую модель YOLO26.
4. Зафиксируйте значения основных метрик.

### Часть C. Сравнение весов
Обучите минимум **3 варианта модели** с разными предобученными весами.  
Например:
- `yolo26n.pt`
- `yolo26s.pt`
- `yolo26m.pt`

Для каждого варианта сравните:
- mAP50;
- mAP50-95;
- precision;
- recall;
- время обучения;
- размер итоговых весов.

### Часть D. Исследование параметров
Проведите не менее **3 дополнительных экспериментов** с параметрами обучения.  
Например, исследуйте влияние:
- `epochs`;
- `imgsz`;
- `batch`;
- `patience`;
- `optimizer`;
- `lr0`.

### Часть E. Анализ результатов
1. Выберите лучшую модель.
2. Выполните инференс на тестовых изображениях.
3. Покажите:
   - 3 примера удачных предсказаний;
   - 3 примера ошибок.
4. Проанализируйте причины ошибок.

## 4. Контрольные вопросы перед выполнением
Ответьте кратко.

1. Что такое задача **object detection** и чем она отличается от **classification**?
2. Что означает **bounding box**?
3. Для чего нужен файл `data.yaml`?
4. Что такое **предобученные веса**?
5. Почему качество разметки критично для качества модели?
6. Что показывают метрики **precision**, **recall**, **mAP50**, **mAP50-95**?

In [1]:
# Запишите здесь краткие ответы на контрольные вопросы.

## 5. Подготовка среды
Согласно туториалу Roboflow, для запуска используются библиотеки:
- `ultralytics`
- `supervision`
- `roboflow`

Также в статье рекомендуется использовать **Google Colab** с включенным **GPU**.

In [ ]:
# При необходимости выполните установку зависимостей
!pip install "ultralytics>=8.4.0" supervision roboflow -q

In [ ]:
# Проверка окружения
import os
import sys
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())

## 6. Подключение датасета из Roboflow

Ниже вставьте код подключения **своего** проекта из Roboflow.  
В статье используется загрузка через API-ключ и вызов `version.download(...)`.

In [ ]:
# Пример-шаблон. Замените значения на свои.
# В Colab удобно хранить ключ в Secrets как ROBOFLOW_API_KEY.

from roboflow import Roboflow

ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace("YOUR_WORKSPACE")
project = workspace.project("YOUR_PROJECT")
version = project.version(YOUR_VERSION_NUMBER)

# Возможный формат экспорта — тот, который поддерживает ваш сценарий YOLO
dataset = version.download("yolov11")

print("Dataset location:", dataset.location)

### Задание 6.1
Укажите параметры вашего датасета:
- тема датасета;
- список классов;
- число изображений;
- число объектов по каждому классу;
- какое разбиение train/valid/test использовано.

In [ ]:
dataset_description = {
    "Тема датасета": "",
    "Классы": [],
    "Число изображений": 0,
    "Число объектов по классам": {},
    "Разбиение": "train/valid/test = "
}

dataset_description

## 7. Базовое обучение модели

В статье в качестве примера используется обучение:
- модель: `yolo26m.pt`
- эпохи: `20`
- размер изображения: `640`

Проведите сначала **базовый запуск**, который затем будет точкой сравнения.

In [ ]:
# Базовое обучение
# При необходимости скорректируйте batch и device под вашу среду.

!yolo task=detect mode=train \
    model=yolo26m.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=640 \
    plots=True

### Задание 7.1
После завершения обучения:
1. Найдите папку с результатами обучения.
2. Укажите путь к:
   - `best.pt`
   - `last.pt`
   - графикам обучения
3. Запишите основные метрики базового запуска.

In [ ]:
baseline_results = {
    "run_dir": "",
    "best_pt": "",
    "last_pt": "",
    "precision": None,
    "recall": None,
    "mAP50": None,
    "mAP50-95": None,
    "notes": ""
}

baseline_results

## 8. Эксперимент 1 — сравнение разных весов

Обучите минимум три варианта модели с разными начальными весами.

### Рекомендуемые варианты
- `yolo26n.pt`
- `yolo26s.pt`
- `yolo26m.pt`

### Что нужно сравнить
- точность по метрикам;
- время обучения;
- устойчивость обучения;
- размер итоговых весов;
- субъективное качество на тестовых изображениях.

In [ ]:
# Примеры запусков. Выполните минимум 3 обучения с разными весами.

# 1) Nano
!yolo task=detect mode=train \
    model=yolo26n.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=640 \
    plots=True \
    project=lab_yolo26 \
    name=exp_weights_n

# 2) Small
!yolo task=detect mode=train \
    model=yolo26s.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=640 \
    plots=True \
    project=lab_yolo26 \
    name=exp_weights_s

# 3) Medium
!yolo task=detect mode=train \
    model=yolo26m.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=640 \
    plots=True \
    project=lab_yolo26 \
    name=exp_weights_m

### Таблица сравнения весов
Заполните таблицу по результатам эксперимента.

In [ ]:
import pandas as pd

weights_comparison = pd.DataFrame([
    {"model": "yolo26n.pt", "epochs": 20, "imgsz": 640, "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "best_pt_size_mb": None, "notes": ""},
    {"model": "yolo26s.pt", "epochs": 20, "imgsz": 640, "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "best_pt_size_mb": None, "notes": ""},
    {"model": "yolo26m.pt", "epochs": 20, "imgsz": 640, "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "best_pt_size_mb": None, "notes": ""},
])

weights_comparison

### Вывод по эксперименту 1
Сделайте вывод:
- какая модель показала лучшее качество;
- какая модель обучалась быстрее;
- какая модель выглядит наиболее рациональной по соотношению **качество / ресурсы**.

In [ ]:
# Ваш вывод по сравнению весов

## 9. Эксперимент 2 — исследование параметров обучения

Проведите минимум три дополнительных запуска, изменяя параметры.
Ниже приведены возможные направления исследования.

### Варианты параметров
1. **Число эпох**: `20`, `50`, `100`
2. **Размер изображения**: `416`, `640`, `960`
3. **Batch size**: `8`, `16`, `32`
4. **Patience** для early stopping
5. **Начальная скорость обучения (`lr0`)**
6. **Оптимизатор**

In [ ]:
# Примеры экспериментальных запусков.
# Выполните любые 3+ запуска и зафиксируйте результаты.

# Эксперимент A: больше эпох
!yolo task=detect mode=train \
    model=yolo26s.pt \
    data={dataset.location}/data.yaml \
    epochs=50 \
    imgsz=640 \
    plots=True \
    project=lab_yolo26 \
    name=exp_params_epochs50

# Эксперимент B: меньший imgsz
!yolo task=detect mode=train \
    model=yolo26s.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=416 \
    plots=True \
    project=lab_yolo26 \
    name=exp_params_img416

# Эксперимент C: другой batch
!yolo task=detect mode=train \
    model=yolo26s.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=640 \
    batch=16 \
    plots=True \
    project=lab_yolo26 \
    name=exp_params_batch16

In [ ]:
params_comparison = pd.DataFrame([
    {"experiment": "epochs=50", "model": "yolo26s.pt", "imgsz": 640, "batch": "default", "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "notes": ""},
    {"experiment": "imgsz=416", "model": "yolo26s.pt", "imgsz": 416, "batch": "default", "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "notes": ""},
    {"experiment": "batch=16", "model": "yolo26s.pt", "imgsz": 640, "batch": 16, "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "notes": ""},
])

params_comparison

### Вывод по эксперименту 2
Ответьте:
1. Какие параметры сильнее всего повлияли на качество?
2. Какие параметры сильнее всего повлияли на скорость?
3. В каких случаях модель начала переобучаться?
4. Какой набор параметров вы считаете оптимальным для вашего датасета?

In [ ]:
# Ваш вывод по исследованию параметров

## 10. Инференс лучшей модели

Выберите лучшую модель и выполните предсказания на тестовых изображениях.
В исходном туториале для инференса используется загрузка `best.pt` и вызов `model.predict(...)`.

In [ ]:
# Пример инференса лучшей модели
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import os

BEST_MODEL_PATH = "PASTE_PATH_TO_BEST_PT"

model = YOLO(BEST_MODEL_PATH)

# Укажите путь к одной или нескольким тестовым картинкам
test_images = [
    # "path/to/test1.jpg",
    # "path/to/test2.jpg",
]

for img_path in test_images:
    result = model.predict(img_path, verbose=False)[0]
    annotated = result.plot()
    plt.figure(figsize=(8, 8))
    plt.imshow(annotated[:, :, ::-1])  # BGR -> RGB
    plt.title(os.path.basename(img_path))
    plt.axis("off")
    plt.show()

### Задание 10.1
Приведите:
- **3 примера удачных детекций**;
- **3 примера неудачных детекций**.

Для каждого неудачного случая укажите причину:
- мало примеров такого объекта в обучении;
- сложный фон;
- маленький размер объекта;
- перекрытие объектов;
- ошибки разметки;
- несбалансированность классов;
- неподходящие параметры обучения.

In [ ]:
successful_cases = [
    {"image": "", "comment": ""},
    {"image": "", "comment": ""},
    {"image": "", "comment": ""},
]

failure_cases = [
    {"image": "", "comment": "", "possible_reason": ""},
    {"image": "", "comment": "", "possible_reason": ""},
    {"image": "", "comment": "", "possible_reason": ""},
]

successful_cases, failure_cases

## 11. Итоговый анализ

Сформулируйте развёрнутый вывод по лабораторной работе.

В выводе отразите:
1. какую задачу обнаружения объектов вы решали;
2. как был собран и размечен датасет;
3. какие веса модели сравнивались;
4. какие параметры изменялись;
5. какая конфигурация оказалась лучшей;
6. какие типичные ошибки делает модель;
7. что можно улучшить в будущем.

In [ ]:
final_conclusion = '''
'''
print(final_conclusion)

## 12. Что должно быть в отчёте

В отчёт необходимо включить:
1. тему и цель работы;
2. описание датасета и классов;
3. скриншоты или ссылку на разметку в Roboflow;
4. таблицу сравнения весов;
5. таблицу сравнения гиперпараметров;
6. итоговые метрики лучшей модели;
7. примеры инференса;
8. анализ ошибок;
9. итоговый вывод.